In [1]:
import pandas as pd
import numpy as np

hab = pd.read_csv('eodgdl/IMEPLAN_Base_Habitantes_Master.csv', encoding='latin1', low_memory=False)
viv = pd.read_csv('eodgdl/IMEPLAN_Base_Viviendas_Master.csv', encoding='latin1', low_memory=False)

print('habitantes:', hab.shape)
print('viviendas:', viv.shape)

habitantes: (58061, 59)
viviendas: (17901, 18)


## Merge Habitantes with Viviendas, keep Habitantes' own weight as FE

In [2]:
tamano_viv_col = 'Incluyéndolo, ¿cuántas personas viven permanentemente en su vivienda contando a los bebés y personas adultas mayores?'

od = hab.merge(
    viv[['Folio Vivienda', tamano_viv_col]],
    on='Folio Vivienda',
    how='left'
)

od = od.rename(columns={'Ponderador': 'FE'})

print(od.shape)

(58061, 60)


## Filter to the employed population (equivalent to ENOE's "p1.coe1 = 1" occupied-only filter)

In [3]:
trabajo_col = 'Durante la semana pasada trabajó:'

od = od[od[trabajo_col].isin(['Tiempo completo', 'Medio tiempo', 'Tenía trabajo, pero no trabajó'])].copy()

print(od.shape)

(26913, 60)


## Recode variables to match `enoe_newvars.py`'s encoding

In [4]:
NO_ESPECIFICADO = 'no_especificado'

od['genero'] = od['Sexo de nacimiento'].map({'Hombres': 'H', 'Mujeres': 'F'})

od['estado_civil'] = od['Estado civil'].map({
    'Soltero': 'soltero',
    'Casado': 'casado',
    'Unión libre': 'union_libre',
    'Viudo': 'viudo',
    'Separado': 'separado',
    'Divorciado': 'divorciado',
    # 'Otros (especifique)' and missing -> left unmapped -> NO_ESPECIFICADO
}).fillna(NO_ESPECIFICADO)

od['parentesco'] = od['Parentesco'].map({
    'Jefe del hogar': 'jefe_del_hogar',
    'Cónyuge': 'conyuge',
    'Compañero': 'conyuge',
    'Hijo': 'hijo',
    'Nieto': 'otro_parentesco',
    'Otro parentesco': 'otro_parentesco',
    'Sin parentesco': 'sin_parentesco',
}).fillna(NO_ESPECIFICADO)

od['municipio'] = od['Municipio'].map({
    'Guadalajara': 'guadalajara',
    'Zapopan': 'zapopan',
    'Tlaquepaque': 'tlaquepaque',
    'Tlajomulco': 'tlajomulco',
    'Tonalá': 'tonala',
    'El Salto': 'el_salto',
    'Juanacatlán': 'otro', # No records in ENOE
    'Ixtlahuacán de los Membrillos': 'ixtlahuacan_membrillos',
    'Zapotlanejo': 'otro', # No records in ENOE
    'Tala': 'tala',
}).fillna('otro')

od['ocupacion'] = od['Ocupación:'].map({
    'Empleado': 'trabajador',
    'Persona trabajadora del hogar': 'trabajador',
    'Trabajador del campo': 'trabajador',
    'Profesor': 'trabajador',
    'Trabajador independiente': 'independiente',
    'Patrón o empresario': 'trabajador', # ENOE code for empleadores (2) is mapped as "trabajador" when processing ENOE
}).fillna("otro") # To match "otro" in ENOE

od['escolaridad'] = od['Escolaridad'].map({
    'Ninguno': 'Sin_Instruccion',
    'Kinder': 'Sin_Instruccion',
    'Preescolar': 'Sin_Instruccion',
    'Primaria': 'Primaria_o_Secundaria',
    'Secundaria': 'Primaria_o_Secundaria',
    'Normal básica': 'Carrera_tecnica_o_preparatoria',
    'Preparatoria o bachillerato': 'Carrera_tecnica_o_preparatoria',
    'Carrera técnica con secundaria terminada': 'Carrera_tecnica_o_preparatoria',
    'Carrera técnica con preparatoria terminada': 'Carrera_tecnica_o_preparatoria',
    'Licenciatura o profesional': 'Licenciatura',
    'Maestría o doctorado': 'Postgrado',
    # 'No sabe' and missing -> left unmapped -> NO_ESPECIFICADO
}).fillna(NO_ESPECIFICADO)

od['sector'] = od['Giro de la empresa donde trabaja:'].map({
    'Servicio': 'Servicios',
    'Educación': 'Servicios',
    'Comercio': 'Comercio',
    'Industria': 'Industria_manufacturera',
    'Gobierno/sector público': 'Gobierno',
}).fillna('Otro')

In [5]:
od['edad_num'] = od['Edad'].astype(int)

od['edad_cat'] = pd.cut(
    od['Edad'],
    (0, 3, 5, 6, 8, 12, 15, 18, 25, 50, 60, 65, 131),
    right=False
).astype(str)


od['tamano_viv_cat'] = od[tamano_viv_col].replace({'10 y +': '10_y_mas'}).fillna(NO_ESPECIFICADO) # OD already reports dwelling size pre-bucketed as "1".."9" plus "10 y +";

## Check before saving


In [6]:
predictor_cols = [
    'genero', 'ocupacion', 'edad_num', 'edad_cat', 'sector',
    'escolaridad', 'municipio', 'estado_civil', 'parentesco',
    'tamano_viv_cat',
]

for c in predictor_cols:
    print(c, '->', od[c].value_counts(dropna=False).to_dict())
    print()

genero -> {'H': 16684, 'F': 10229}

ocupacion -> {'trabajador': 23274, 'independiente': 3417, 'otro': 222}

edad_num -> {31: 1158, 41: 1095, 26: 843, 51: 818, 46: 814, 33: 805, 39: 800, 29: 767, 36: 753, 30: 750, 43: 699, 28: 698, 37: 689, 40: 686, 27: 676, 38: 669, 21: 668, 24: 666, 23: 649, 25: 648, 34: 608, 49: 586, 32: 543, 44: 530, 53: 519, 48: 491, 50: 470, 22: 461, 47: 449, 45: 438, 19: 427, 20: 426, 42: 423, 54: 417, 61: 404, 56: 399, 57: 362, 59: 342, 58: 318, 52: 304, 60: 251, 63: 204, 64: 204, 66: 199, 35: 176, 18: 154, 62: 149, 65: 129, 17: 115, 67: 103, 68: 98, 69: 95, 16: 88, 71: 83, 55: 74, 15: 70, 13: 60, 70: 59, 14: 57, 12: 56, 73: 39, 72: 32, 74: 27, 76: 23, 77: 17, 79: 15, 75: 14, 80: 14, 78: 14, 81: 9, 84: 5, 87: 3, 86: 2, 83: 2, 82: 2, 89: 1, 85: 1, 95: 1}

edad_cat -> {'[25, 50)': 16794, '[50, 60)': 4023, '[18, 25)': 3451, '[60, 65)': 1212, '[65, 131)': 987, '[15, 18)': 273, '[12, 15)': 173}

sector -> {'Otro': 9484, 'Servicios': 6372, 'Comercio': 5868, 'Industria

## Assemble final dataset and save

In [7]:
od_processed = od[predictor_cols + ['FE']].reset_index(drop=True)

print(od_processed.shape)
od_processed.to_csv('od_processed.csv', index=False)
od_processed

(26913, 11)


,genero,ocupacion,edad_num,edad_cat,sector,escolaridad,municipio,estado_civil,parentesco,tamano_viv_cat,FE
0,H,trabajador,51,"[50, 60)",Servicios,Postgrado,tlajomulco,casado,jefe_del_hogar,5,51
1,H,trabajador,68,"[65, 131)",Servicios,Primaria_o_Secundaria,guadalajara,soltero,jefe_del_hogar,3,249
2,H,trabajador,61,"[60, 65)",Servicios,Carrera_tecnica_o_preparatoria,guadalajara,soltero,otro_parentesco,3,249
3,H,trabajador,76,"[65, 131)",Servicios,Licenciatura,guadalajara,casado,conyuge,3,249
4,F,independiente,63,"[60, 65)",Comercio,Primaria_o_Secundaria,guadalajara,casado,conyuge,3,249
...,...,...,...,...,...,...,...,...,...,...,...
26908,H,trabajador,20,"[18, 25)",Industria_manufacturera,Primaria_o_Secundaria,tlaquepaque,soltero,otro_parentesco,4,178
26909,F,trabajador,30,"[25, 50)",Industria_manufacturera,Primaria_o_Secundaria,tlaquepaque,casado,conyuge,6,178
26910,H,trabajador,33,"[25, 50)",Otro,Primaria_o_Secundaria,tlaquepaque,casado,jefe_del_hogar,6,178
26911,H,independiente,66,"[65, 131)",Comercio,Primaria_o_Secundaria,tlaquepaque,casado,jefe_del_hogar,2,178
